# Cellbender (@ Terra workspace)

In [ ]:
fpr = 0.01

cellbender remove-background \
      --input "${input_10x_h5_file_or_mtx_directory}" \
      --output "${sample_name}_out.h5" \
      --cuda \
      ${"--expected-cells " + expected_cells} \
      ${"--total-droplets-included " + total_droplets_included} \
      ${"--fpr " + fpr} \

# Doublet scoring

In [ ]:
import os
import sys
import argparse
hashseed = os.getenv('PYTHONHASHSEED')
if not hashseed:
    os.environ['PYTHONHASHSEED'] = '0'
    os.execv(sys.executable, [sys.executable] + sys.argv)

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scrublet as scr
import seaborn as sns
import matplotlib.pyplot as plt

import anndata2ri
import rpy2.robjects as ro
import rpy2.robjects.conversion as cv
anndata2ri.activate()


## install
#ro.r('install.packages("BiocManager", ' 'repos="http://cran.r-project.org")')
#packages_to_install = ['scuttle', 'scran', 'scater', 'bluster', 'scDblFinder', 'scds']
#for package in packages_to_install:
#    ro.r(f'BiocManager::install("{package}")')

def run_scDblFinder(adata):
    """
    run three methods recommended in scDblFinder
    """
    r_code_scDblFinder = """
    r_scDblFinder = function(sce) {
      library(scuttle)
      library(scran)
      library(scater)
      library(bluster)
      library(scDblFinder)
  
      # method 1: computeDoubletDensity
      # https://bioconductor.org/packages/release/bioc/vignettes/scDblFinder/inst/doc/computeDoubletDensity.html
      sce = logNormCounts(sce)
      hvgs = getTopHVGs(modelGeneVar(sce), n=2000)
      set.seed(1001)
      sce = runPCA(sce, ncomponents=10, subset_row=hvgs)
      sce = runTSNE(sce, dimred="PCA")
      set.seed(1002)
      m1.score = computeDoubletDensity(sce, subset.row=hvgs)
 
      # method 2: findDoubletCluster
      # https://bioconductor.org/packages/release/bioc/vignettes/scDblFinder/inst/doc/findDoubletClusters.html
      clusters = clusterRows(reducedDim(sce, "PCA"), NNGraphParam())
      tab = findDoubletClusters(sce, clusters)
      m2.class = ifelse(tab[as.character(clusters),]$p.value > 0.05, "doublet", "singlet")

      # method 3: scDblFinder
      # https://bioconductor.org/packages/release/bioc/vignettes/scDblFinder/inst/doc/scDblFinder.html
      sce = scDblFinder(sce)
      m3.score = sce$scDblFinder.score
      m3.class = as.character(sce$scDblFinder.class)

      df = data.frame(index=colnames(sce),
                      dbl_cDD_score=m1.score,
                      dbl_fDC_class=m2.class,
                      dbl_scDF_score=m3.score,
                      dbl_scDF_class=m3.class,
                      row.names=1)
      return(df)
    } """
    r_scDblFinder = ro.r(r_code_scDblFinder)
    try:
        df = r_scDblFinder(adata)
        return(df)
    except:
        d = {'dbl_cDD_score' : [np.nan for i in adata.obs.index],
             'dbl_fDC_class' : [np.nan for i in adata.obs.index],
             'dbl_scDF_score': [np.nan for i in adata.obs.index],
             'dbl_scDF_class': [np.nan for i in adata.obs.index]}
        return pd.DataFrame(data=d, index=adata.obs.index)


def run_scds(adata):
    """
    run SCDS and compute three types of doublet scores
    """
    colnames = {'cxds_score'  :'dbl_cxds_score',
                'bcds_score'  :'dbl_bcds_score',
                'hybrid_score':'dbl_hybrid_score'}
    try:
        scds_r = ro.packages.importr('scds')
        ro.r['set.seed'](1001)
        x = scds_r.cxds_bcds_hybrid(adata)
        x = x.obs[['cxds_score', 'bcds_score', 'hybrid_score']]
        return(x.rename(columns=colnames))
    except:
        d = {'dbl_cxds_score'  : [np.nan for i in adata.obs.index],
             'dbl_bcds_score'  : [np.nan for i in adata.obs.index],
             'dbl_hybrid_score': [np.nan for i in adata.obs.index]}
        return pd.DataFrame(data=d, index=adata.obs.index)


def run_scrublet(adata, sample_id=None, cutoff=None):
    """
    run scrublet and compute doublet scores
    save QC plots for manual review when sample_id is supplied
    """
    scrub = scr.Scrublet(adata.X, expected_doublet_rate=0.06)
    try:
        scores, flags = scrub.scrub_doublets(min_counts=2, 
                                             min_cells=3, 
                                             min_gene_variability_pctl=85, 
                                             n_prin_comps=30)
        # update flags if a custom cutoff is supplied
        if cutoff != None:
            flags = scrub.call_doublets(threshold=cutoff)
            
        if sample_id != None:
            # save outputs for manual review
            fo = open(sample_id+'.scrublet.log', 'w')
            fo.write('auto_threshold={}\n'.format(scrub.threshold_))
            fo.write('detected_doublet_rate={}\n'.format(scrub.detected_doublet_rate_))
            fo.write('detectable_doublet_fraction={}\n'.format(scrub.detectable_doublet_fraction_))
            fo.write('overall_doublet_rate={}\n'.format(scrub.overall_doublet_rate_))
            fo.close()
            scrub.plot_histogram()
            plt.savefig(sample_id+'.scrublet_histogram.png', dpi=300)
            scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
            scrub.plot_embedding('UMAP', order_points=True)
            plt.savefig(sample_id+'.scrublet_umap.png', dpi=300)

        d = {'dbl_scrublet_score': scores,
             'dbl_scrublet_class': np.where(flags==True,'doublet','singlet') }
        return pd.DataFrame(data=d, index=adata.obs.index)
    except:
        d = {'dbl_scrublet_score': [np.nan for i in adata.obs.index],
             'dbl_scrublet_class': [np.nan for i in adata.obs.index]}
        return pd.DataFrame(data=d, index=adata.obs.index)
    

def doublet_tools(adata, sample_id=None):
    adata.layers['counts'] = adata.X.copy()
    scds_ = run_scds(adata)
    scdbl_ = run_scDblFinder(adata)
    scrub_ = run_scrublet(adata, sample_id=sample_id)
    bc = scds_.index
    return pd.concat([scds_.loc[bc], scdbl_.loc[bc], scrub_.loc[bc]], axis=1)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Run doublet tools')
    parser.add_argument('h5ad_path', help='AnnData h5ad path')
    args = parser.parse_args()
    try:
        sample_id = os.path.basename(args.h5ad_path).split('.h5')[0]  ##.split('.h5ad')[0]
        adata = sc.read_10x_h5(args.h5ad_path)     ##sc.read_h5ad   
        dbl = doublet_tools(adata, sample_id)
        dbl.to_csv(sample_id+'.dbl.csv')
    except KeyboardInterrupt:
        pass


# Merge and MT filter

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
#import scanpy.external as sce
#import os, glob
import anndata as ad
import sys #, subprocess

sample=sys.argv[1]
res='0.1'

list = []
for x in open(sys.argv[2], 'r').readlines():
#    name, file = x.rstrip().split("\t")
    file = x.rstrip()
    name = file.split("/")[-1].split("_cellbender")[0]
    adata = sc.read_h5ad(file)
    adata.var_names_make_unique()
    adata.obs['library'] = np.array(name)
    adata.obs.index = np.array(adata.obs.index) + '-' + np.array(name)

    for col in adata.obs.columns:
        if col.startswith('dbl_') and 'score' in col:
            adata.obs[col] = pd.to_numeric(adata.obs[col], errors='coerce').astype(float)

        elif adata.obs[col].dtype == object:
            adata.obs[col] = adata.obs[col].astype(str)

    list.append(adata)

adata = ad.concat(list)
#adata.write_h5ad('merged_' + sample + '_raw.h5ad')

#### Preprocessing
## Filter cells with mtgenes with more than 15% 
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs.pct_counts_mt < 15, :]

sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save = '_' + sample + '_total_count_ngc.pdf')

adata.write_h5ad('merged_' + sample + '_raw.h5ad')

# Extract B-cell or PC cluster 

In [ ]:
adata_ori = sc.read_h5ad('merged_' + sample + '_raw.h5ad')

for pid in pid_list:
    print(f'************** {pid} **************')
    
    adata_ori.obs['PROMISE_ID'] = adata_ori.obs['library'].map(demux_dict)
    
    adata = adata_ori[adata_ori.obs['PROMISE_ID'] == pid]
    
    adata = adata[adata.obs['library'].isin(vdj_present_libraries)]

    #### Mask genes (IG, and MT genes)
    sex_specific_malat1_genes = ['XIST', 'RPS4Y1', 'DDX3Y', 'MALAT1']
    
    mask = (
        adata.var_names.str.startswith('IG') | 
        adata.var_names.str.startswith('MT-') | 
        adata.var_names.str.startswith('RPL') | 
        adata.var_names.str.startswith('RPS') | 
        adata.var_names.isin(sex_specific_malat1_genes)
    )
    adata = adata[:, ~mask]

    #### Total-count normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    #scale data, clip values exceeding standard deviation 10.
    sc.pp.scale(adata, max_value=10)

    res=0.2

    #### Run PCA and UMAP
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
    sc.tl.leiden(adata, resolution=float(res)) #sc.tl.lougvain(adata)

    sc.tl.paga(adata)
    sc.pl.paga(adata, plot=False)  # remove `plot=False` if you want to see the coarse-grained graph
    sc.tl.umap(adata, init_pos='paga')


    ### Visualization

    sc.pl.embedding(adata, basis='X_umap', 
                    color=['library'], 
                    frameon=False)

    ## cell quality
    sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'])

    ## doublet
    dbl_list = ['dbl_cxds_score', 'dbl_bcds_score', 'dbl_hybrid_score', 'dbl_cDD_score', 'dbl_fDC_class', 'dbl_scDF_score', 'dbl_scDF_class', 'dbl_scrublet_score', 'dbl_scrublet_class']
    dbl_score_list = [x for x in dbl_list if 'score' in x]
    dbl_score_valid_list=[]
    for dbl in dbl_score_list:
        value = pd.to_numeric(adata.obs[dbl], errors='coerce')
        if np.any(~np.isnan(value)):
            dbl_score_valid_list.append(dbl)
        adata.obs[dbl] = adata.obs[dbl].astype(float)

    sc.pl.umap(adata, use_raw=False, color=dbl_score_valid_list, color_map='viridis')
    #################################### Check cells
    sc.pl.umap(adata, color=['leiden'], legend_loc='on data')
    # ## monocyte (LYZ), T cell (IL7R, NKG7), Macrophage (C1QA, C1QB), RBC (HBA, HBB)
    sc.pl.umap(adata, color=['LYZ', 'IL7R', 'NKG7', 'C1QA', 'C1QB', 'HBA2', 'HBB'])
    ## plasma
    sc.pl.umap(adata, color=['MZB1', 'CD38', 'XBP1'])
    ## B cell
    sc.pl.umap(adata, color=['CD79A', 'CD79B', 'CD19'])

    target_genes = ['CCND1', 'FGFR3', 'WHSC1', 'MAF', 'MAFB', 'CCND3']
    ## t(11;14): CCND1, t(4;14): FGFR3, MMSET, t(14;16): MAF, t(14;20): MAFB, t(6;14): CCND3
    target_genes_valid = [gene for gene in target_genes if gene in adata.var_names]

    sc.pl.embedding(adata, basis='X_umap', 
        color=target_genes_valid, frameon=False)
    
    
####################################################### After manual inspection of finding B-cell/PC clusters
## PC/B-cell
# res=0.2
pid_cluster_dict = {
    1329 : [0,3,12,13,      6,1], # PC + B cell 
    2115 : [8,16,    3,4,5,], # PC+B,  B 
    2936 : [8,9,     0,1,3],  # PC, B
    2512 : [1], #B 
    563 : [9,      0,5,6], # PC, B 
    1106 : [11,    0,3,9,2],  #PC, B
    1263 : [8,   3,11], #PC, B 
    6373 : [1,3,7], # B
}



adata_master = adata_ori.copy()
adata_master.obs['PROMISE_ID'] = adata_master.obs['library'].map(demux_dict)

final_selected_barcodes = []

for pid in pid_cluster_dict.keys():
    print(f'************** Processing PID: {pid} **************')
    adata = adata_master[adata_master.obs['PROMISE_ID'] == pid].copy()
    
    adata = adata[adata.obs['library'].isin(vdj_present_libraries)]

    #### Mask genes (IG, and MT genes)
    sex_specific_malat1_genes = ['XIST', 'RPS4Y1', 'DDX3Y', 'MALAT1']
    
    mask = (
        adata.var_names.str.startswith('IG') | 
        adata.var_names.str.startswith('MT-') | 
        adata.var_names.str.startswith('RPL') | 
        adata.var_names.str.startswith('RPS') | 
        adata.var_names.isin(sex_specific_malat1_genes)
    )
    adata = adata[:, ~mask]

    #### Total-count normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    #scale data, clip values exceeding standard deviation 10.
    sc.pp.scale(adata, max_value=10)

    res=0.2

    #### Run PCA and UMAP
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
    sc.tl.leiden(adata, resolution=float(res)) #sc.tl.lougvain(adata)

    sc.tl.paga(adata)
    sc.pl.paga(adata, plot=False)  # remove `plot=False` if you want to see the coarse-grained graph
    sc.tl.umap(adata, init_pos='paga')
    
    target_clusters = pid_cluster_dict.get(pid, [])
    
    if len(target_clusters) > 0:
            str_clusters = [str(c) for c in target_clusters]
            selected_indices = adata.obs_names[adata.obs['leiden'].isin(str_clusters)].tolist()
            final_selected_barcodes.extend(selected_indices)
            print(f"PID {pid}: Extracted {len(selected_indices)} cells out of {adata.n_obs}")
    else:
        final_selected_barcodes.extend(adata.obs_names.tolist())
        print(f"PID {pid}: Added all {adata.n_obs} barcodes")

adata_master_filtered = adata_master[adata_master.obs_names.isin(final_selected_barcodes)].copy()

adata_master_filtered.write(save_path)
